# 02: Patching Basics

Learn how to use activation patching to test causal hypotheses about your world model.

In [ ]:
from world_model_lens import HookedWorldModel, WorldModelConfig, ActivationCache
from world_model_lens.patching.patcher import TemporalPatcher, PatchResult
import torch

## Setup World Model

In [ ]:
# Create hooked world model (reuse from 01_quickstart)
wm = HookedWorldModel(adapter=None, config=WorldModelConfig())

# Generate clean and corrupted trajectories
clean_obs = torch.randn(20, 128)
clean_actions = torch.randn(20, 4)

corrupted_obs = torch.randn(20, 128) * 2  # Double the noise
corrupted_actions = torch.randn(20, 4)

## Collect Activation Caches

In [ ]:
clean_traj, clean_cache = wm.run_with_cache(clean_obs, clean_actions)
corrupted_traj, corrupted_cache = wm.run_with_cache(corrupted_obs, corrupted_actions)

print(f"Clean cache components: {clean_cache.component_names}")

## Define Metric and Run Patching

In [ ]:
# Define metric: mean hidden state similarity
def hidden_state_metric(traj):
    return traj.h_sequence.mean().item()

# Create patcher
patcher = TemporalPatcher(wm)

# Run full sweep
components = ['encoder', 'gru', 'posterior', 'prior', 'reward']
results = patcher.full_sweep(
    clean_cache=clean_cache,
    corrupted_cache=corrupted_cache,
    components=components,
    metric_fn=hidden_state_metric,
)

## Analyze Results

In [ ]:
# Get top patches
top_patches = results.top_k_patches(k=5)
print("Top 5 patches by recovery rate:")
for p in top_patches:
    print(f"  {p.component}: {p.recovery_rate:.3f}")

In [ ]:
# Visualize recovery matrix
import matplotlib.pyplot as plt

recovery_matrix = results.recovery_matrix()
plt.imshow(recovery_matrix, aspect='auto', cmap='RdYlGn')
plt.colorbar(label='Recovery Rate')
plt.xlabel('Timestep')
plt.ylabel('Component')
plt.title('Activation Patching Recovery')
plt.show()